# Homework 02

Your task is to train a model that will improve upon at least one of the baselines provided
- Any classic ML model can be used (**not** neural networks)
- You can engineer more features if you want, e.g. holidays
- You can use feature selection methods not from `tsfresh`

In [ ]:
!pip install numpy pandas matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd

import pickle

import IPython.display as dp
import matplotlib.pyplot as plt
import seaborn as sns

dp.set_matplotlib_formats("retina")
sns.set(style="whitegrid", font_scale=1.5)
sns.despine()

%matplotlib inline

## Load data

Please choose one stock ticker of your liking.

In [ ]:
with open("returns_NASDAQ_hist_daily_06102021-091329.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
data["Ticker"].unique()

In [ ]:
MY_TICKER = # YOUR CHOICE

In [ ]:
data_ticker = data[data["Ticker"] == MY_TICKER]

In [ ]:
data_ticker = data_ticker.sort_index().reset_index()

In [ ]:
data_ticker.head()

In [ ]:
fig, ax = plt.subplots(figsize=(16,9))
data_ticker.plot(x="Time", y="Close", ax=ax)

## Create features

We will be working with classic machine learning methods in this assignment, so we must supply fixed sized input. In order to work with sequences of variable size within this setting, windowing is used. `tsfresh` provides windowing through `roll_time_series` function. Please familiarize yourself with its arguments and make sure to understand the operation it performs on the data.

In [ ]:
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import roll_time_series, make_forecasting_frame
from tsfresh.utilities.dataframe_functions import impute

In [ ]:
data_rolled = roll_time_series(
    data_ticker,
    column_id="Ticker",
    column_sort="Time",
    max_timeshift=20,
    min_timeshift=5,
)

In [ ]:
data_rolled.head()

Now that we have inputs of equal size (one `id` is one sample), we can create features for them using `tsfresh.extract_features`. Please familiarize yourself with its arguments.

In [ ]:
X = extract_features(
    data_rolled.drop("Ticker", axis=1).fillna(0),
    column_id="id",X.
    column_sort="Time",
    column_value="Close",
    impute_function=impute,
    show_warnings=False
)

In [ ]:
X = X.set_index(X.index.map(lambda x: x[1]), drop=True)
X.index.name = "last_date"
X.shape

In [ ]:
X.columns

In [ ]:
y = data_ticker.set_index("Time").sort_index()['Close'].shift(-1).dropna()

In [ ]:
common_index = X.index.intersection(y.index)
X = X.loc[common_index]
y = y.loc[common_index]

In [ ]:
X_train = X[:"2018"]
X_test = X["2019":]

y_train = y[:"2018"]
y_test = y["2019":]

## Select features

`tsfresh` generates too many features for classic ML models that suffer from curse of dimensionality, so we need to choose the most representative features. `tsfresh` provides a function `select_features` for this purpose. You can use this function, or you can use PCA or any other feature selection / dimensionality reduction method that you know. If you choose to not use `tsfresh.select_features`, please still familiarize yourself with its arguments.

In [ ]:
from tsfresh import select_features

In [ ]:
X_train_selected = select_features(...)

In [ ]:
X_test_selected = X_test[X_train_selected.columns]

In [ ]:
X_train_selected.shape

## Preprocess features

You are welcome to perform some preprocessing steps that may improve the quality of your model. Please note that evaluation must be done on the original features.

In [ ]:
from sklearn.preprocessing import StandardScaler

## Baselines

I have created three baselines for you. Your task will be to beat at least the first one. If you beat the rolling mean baseline, you will get bonus points. You are also free to introduce our own baselines, e.g. linear regression.

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
mean_squared_error(y_test, y_test.shift(1).fillna(0))

In [ ]:
mean_squared_error(y_test, np.ones_like(y_test) * y_test.mean())

In [ ]:
mean_squared_error(y_test, y_test.rolling(window=3, center=True).mean().bfill().ffill())

## Train a model

- Pick any ML model of your liking
- Fit it to the selected (and possibly preprocessed) features on the train set
- You can do cross-validation to pick hyperparameters, but be careful not to train on future!
- Make a prediction on the test set
- If necessay, make an inverse transform

## Compare quality

Please make a ualitative (plots) and quantitative (quality measures) comparison between your model and the baselines.